# MVC (Model-View-Controller): exemplo executável

**Guia Aberto de Arquitetura de Software** · [Página do tópico](https://ifpe-belojardim-es.github.io/guia-arquitetura-de-software/content/01-mvc/)

Autores: João Gabriel, David Soares, João Lucas, Ian Kepler, Lucas Matheus, Antonio Mario · Licença: CC BY-NC 4.0

---

Neste notebook a gente monta o mesmo sistema de duas formas:

1. **Seguindo o MVC**: a regra de aprovação fica só no Model, e as telas apenas mostram.
2. **Pegando um atalho**: cada tela calcula a média e decide a aprovação por conta própria.

Depois mudamos a média mínima de 6,0 para 7,0 e vemos o que acontece em cada versão.

## 0. Dependências

Nada para instalar. Tudo aqui usa só a biblioteca padrão do Python, então roda igual no Colab e em qualquer máquina com Python 3.9 ou mais novo.

In [1]:
import sys
import dis
from typing import Callable, Dict, List

print(f"Python {sys.version.split()[0]}")

Python 3.11.15


## 1. O problema

Um professor quer acompanhar o **boletim da turma**. Ele lança notas e precisa de duas visões:

- uma **tabela** com cada aluno, sua média e a situação (aprovado ou reprovado);
- um **resumo** dizendo quantos alunos passaram e quantos ficaram.

A regra de hoje: média maior ou igual a **6,0** aprova. Só que regra de escola muda, e a gente quer que essa mudança seja feita em um lugar só.

## 2. Versão MVC

### 2.1 O Model

A `Turma` guarda as notas, calcula a média e decide quem está aprovado. Ela também mantém uma lista de **observadores**: qualquer função que queira ser avisada quando algo mudar.

Repare que a classe não importa nada de tela e não chama `print`. Ela não sabe se vai ser mostrada no terminal, numa página web ou se está dentro de um teste.

In [2]:
class Turma:
    """Model: dados e regras de negócio do boletim."""

    def __init__(self, nome: str, media_minima: float = 6.0):
        self.nome = nome
        self.media_minima = media_minima
        self._notas: Dict[str, List[float]] = {}
        self._observadores: List[Callable[["Turma"], None]] = []

    # --- mecanismo de aviso (padrão Observer) ---
    def inscrever(self, observador: Callable[["Turma"], None]) -> None:
        self._observadores.append(observador)

    def _avisar(self) -> None:
        for observador in self._observadores:
            observador(self)

    # --- operações que mudam o estado ---
    def matricular(self, aluno: str) -> None:
        self._notas.setdefault(aluno, [])
        self._avisar()

    def lancar_nota(self, aluno: str, nota: float) -> None:
        if not 0 <= nota <= 10:
            raise ValueError("A nota precisa estar entre 0 e 10")
        if aluno not in self._notas:
            raise KeyError(f"{aluno} não está matriculado")
        self._notas[aluno].append(nota)
        self._avisar()

    def mudar_media_minima(self, nova: float) -> None:
        self.media_minima = nova
        self._avisar()

    # --- consultas (a regra de negócio mora aqui) ---
    def alunos(self) -> List[str]:
        return sorted(self._notas)

    def media(self, aluno: str) -> float:
        notas = self._notas[aluno]
        return sum(notas) / len(notas) if notas else 0.0

    def aprovado(self, aluno: str) -> bool:
        return self.media(aluno) >= self.media_minima

### 2.2 As Views

Cada View recebe a turma e mostra do seu jeito. Nenhuma delas faz conta de aprovação: elas **perguntam** ao Model (`turma.aprovado(aluno)`).

In [3]:
class TabelaView:
    """View 1: tabela completa, uma linha por aluno."""

    def __init__(self):
        self.ultima_saida = {}

    def atualizar(self, turma: Turma) -> None:
        self.ultima_saida = {}
        print(f"\n[Tabela] {turma.nome} (média mínima {turma.media_minima:.1f})")
        for aluno in turma.alunos():
            situacao = "aprovado" if turma.aprovado(aluno) else "reprovado"
            self.ultima_saida[aluno] = situacao
            print(f"  {aluno:<8} média {turma.media(aluno):4.1f}  {situacao}")


class ResumoView:
    """View 2: só os números da turma."""

    def __init__(self):
        self.aprovados = []

    def atualizar(self, turma: Turma) -> None:
        self.aprovados = [a for a in turma.alunos() if turma.aprovado(a)]
        total = len(turma.alunos())
        print(f"[Resumo] {len(self.aprovados)} de {total} aprovados")

### 2.3 O Controller

O Controller recebe o que o professor digita, entende o comando e chama o método certo do Model. Ele **não sabe** que existem duas Views: quem avisa as telas é o próprio Model.

In [4]:
class BoletimController:
    """Controller: traduz comandos de texto em operações no Model."""

    def __init__(self, turma: Turma):
        self.turma = turma

    def executar(self, comando: str) -> None:
        partes = comando.split()
        acao = partes[0]
        if acao == "matricular":
            self.turma.matricular(partes[1])
        elif acao == "nota":
            self.turma.lancar_nota(partes[1], float(partes[2]))
        elif acao == "media_minima":
            self.turma.mudar_media_minima(float(partes[1]))
        else:
            print(f"Comando desconhecido: {acao}")

### 2.4 Juntando as peças

Criamos a turma, inscrevemos as duas Views e mandamos alguns comandos pelo Controller. Para não poluir a saída, as Views só são inscritas depois das matrículas.

In [5]:
turma = Turma("ES 2026.2")
controller = BoletimController(turma)

for aluno in ["Ana", "Bruno", "Carla"]:
    controller.executar(f"matricular {aluno}")

tabela = TabelaView()
resumo = ResumoView()
turma.inscrever(tabela.atualizar)
turma.inscrever(resumo.atualizar)

controller.executar("nota Ana 8.0")
controller.executar("nota Bruno 5.0")
controller.executar("nota Carla 6.5")


[Tabela] ES 2026.2 (média mínima 6.0)
  Ana      média  8.0  aprovado
  Bruno    média  0.0  reprovado
  Carla    média  0.0  reprovado
[Resumo] 1 de 3 aprovados

[Tabela] ES 2026.2 (média mínima 6.0)
  Ana      média  8.0  aprovado
  Bruno    média  5.0  reprovado
  Carla    média  0.0  reprovado
[Resumo] 1 de 3 aprovados

[Tabela] ES 2026.2 (média mínima 6.0)
  Ana      média  8.0  aprovado
  Bruno    média  5.0  reprovado
  Carla    média  6.5  aprovado
[Resumo] 2 de 3 aprovados


Cada nota lançada fez as duas telas se redesenharem, e o Controller nem sabe que elas existem.

### 2.5 A escola mudou a regra

Agora a média mínima passa a ser **7,0**. No MVC isso é **um** comando, e a mudança chega às duas Views de uma vez, porque as duas perguntam ao mesmo Model.

In [6]:
controller.executar("media_minima 7.0")

assert tabela.ultima_saida["Carla"] == "reprovado"
assert "Carla" not in resumo.aprovados
print("\nAs duas telas concordam: Carla (6,5) está reprovada.")


[Tabela] ES 2026.2 (média mínima 7.0)
  Ana      média  8.0  aprovado
  Bruno    média  5.0  reprovado
  Carla    média  6.5  reprovado
[Resumo] 1 de 3 aprovados

As duas telas concordam: Carla (6,5) está reprovada.


### 2.6 Testando a regra sem nenhuma tela

Como a regra está só no Model, dá para testá-la sem criar View nem Controller. Isso é o que torna o MVC testável na prática.

In [7]:
t = Turma("teste", media_minima=6.0)
t.matricular("Zé")
t.lancar_nota("Zé", 6.0)
assert t.aprovado("Zé")

t.mudar_media_minima(7.0)
assert not t.aprovado("Zé")

print("Testes do Model passaram, sem nenhuma tela envolvida.")

Testes do Model passaram, sem nenhuma tela envolvida.


## 3. A violação: cada tela com a sua regra

Agora a versão "atalho". Não existe Model de verdade, só um dicionário de notas. Cada tela calcula a média e decide a aprovação sozinha.

Parece mais simples, porque tem menos classes. Vamos ver o que acontece quando a regra muda.

In [8]:
notas = {"Ana": [8.0], "Bruno": [5.0], "Carla": [6.5]}


class TabelaAtalho:
    def mostrar(self, notas):
        saida = {}
        print("[Tabela atalho]")
        for aluno, ns in sorted(notas.items()):
            media = sum(ns) / len(ns)
            situacao = "aprovado" if media >= 6.0 else "reprovado"   # regra aqui
            saida[aluno] = situacao
            print(f"  {aluno:<8} média {media:4.1f}  {situacao}")
        return saida


class ResumoAtalho:
    def mostrar(self, notas):
        aprovados = [a for a, ns in notas.items()
                     if sum(ns) / len(ns) >= 6.0]                    # e aqui de novo
        print(f"[Resumo atalho] {len(aprovados)} de {len(notas)} aprovados")
        return aprovados


TabelaAtalho().mostrar(notas)
ResumoAtalho().mostrar(notas);

[Tabela atalho]
  Ana      média  8.0  aprovado
  Bruno    média  5.0  reprovado
  Carla    média  6.5  aprovado
[Resumo atalho] 2 de 3 aprovados


Por enquanto tudo certo. Agora a escola muda a média para **7,0**. Alguém do grupo atualiza a tabela, mas esquece do resumo, que fica em outro arquivo. Isso é bem comum: quem mexeu na tabela nem sabia que o resumo repetia a mesma conta.

In [9]:
class TabelaAtalhoV2:
    """Tabela atualizada para a nova regra."""
    def mostrar(self, notas):
        saida = {}
        print("[Tabela atalho v2]")
        for aluno, ns in sorted(notas.items()):
            media = sum(ns) / len(ns)
            situacao = "aprovado" if media >= 7.0 else "reprovado"   # mudou aqui
            saida[aluno] = situacao
            print(f"  {aluno:<8} média {media:4.1f}  {situacao}")
        return saida


saida_tabela = TabelaAtalhoV2().mostrar(notas)
aprovados_resumo = ResumoAtalho().mostrar(notas)                     # esqueceram deste

print()
print(f"Tabela diz que Carla está: {saida_tabela['Carla']}")
print(f"Resumo conta Carla como aprovada? {'Carla' in aprovados_resumo}")
if (saida_tabela["Carla"] == "reprovado") and ("Carla" in aprovados_resumo):
    print("\n>> INCONSISTÊNCIA: o mesmo sistema dá duas respostas para a mesma aluna.")

[Tabela atalho v2]
  Ana      média  8.0  aprovado
  Bruno    média  5.0  reprovado
  Carla    média  6.5  reprovado
[Resumo atalho] 2 de 3 aprovados

Tabela diz que Carla está: reprovado
Resumo conta Carla como aprovada? True

>> INCONSISTÊNCIA: o mesmo sistema dá duas respostas para a mesma aluna.


## 4. Consequência mensurável

Duas medidas simples:

1. **Quantos lugares decidem a aprovação?** Procuramos, no bytecode de cada classe, comparações com a média mínima (o `>=`). Isso funciona no Colab sem precisar ler o arquivo-fonte.
2. **Dá para testar a regra sem tela?** Na versão MVC, sim (já fizemos na seção 2.6). Na versão atalho, a regra só existe dentro dos métodos `mostrar`, então testar a regra é testar a tela.

In [10]:
def compara_media(cls) -> bool:
    """True se algum método da classe faz uma comparação >= (a regra de aprovação),
    olhando também para funções internas como list comprehensions."""
    def percorre(codigo):
        for instr in dis.get_instructions(codigo):
            if instr.opname == "COMPARE_OP" and ">=" in str(instr.argval):
                return True
        return any(percorre(c) for c in codigo.co_consts if hasattr(c, "co_code"))

    for membro in vars(cls).values():
        codigo = getattr(membro, "__code__", None)
        if codigo is not None and percorre(codigo):
            return True
    return False


versoes = {
    "MVC":    [Turma, TabelaView, ResumoView, BoletimController],
    "Atalho": [TabelaAtalhoV2, ResumoAtalho],
}

for nome, classes in versoes.items():
    com_regra = [c.__name__ for c in classes if compara_media(c)]
    print(f"{nome:<7} lugares com a regra de aprovação: {len(com_regra)}  -> {', '.join(com_regra)}")

MVC     lugares com a regra de aprovação: 1  -> Turma
Atalho  lugares com a regra de aprovação: 2  -> TabelaAtalhoV2, ResumoAtalho


E o custo cresce com o sistema. Se aparecer uma terceira tela (um relatório em PDF, uma API para o app), a versão atalho ganha mais um lugar com a regra. A versão MVC continua com um.

In [11]:
for telas in range(1, 6):
    print(f"{telas} tela(s): MVC = 1 lugar para mudar | atalho = {telas} lugar(es) para mudar")

1 tela(s): MVC = 1 lugar para mudar | atalho = 1 lugar(es) para mudar
2 tela(s): MVC = 1 lugar para mudar | atalho = 2 lugar(es) para mudar
3 tela(s): MVC = 1 lugar para mudar | atalho = 3 lugar(es) para mudar
4 tela(s): MVC = 1 lugar para mudar | atalho = 4 lugar(es) para mudar
5 tela(s): MVC = 1 lugar para mudar | atalho = 5 lugar(es) para mudar


## 5. Conclusão

Na versão MVC, mudar a média mínima foi um comando e as duas telas concordaram. Na versão atalho, a mesma mudança deixou o sistema contraditório, com a tabela reprovando e o resumo aprovando a mesma aluna. Esse é o ganho de **modificabilidade** e **testabilidade** discutido no texto. O preço é ter mais classes e um fluxo menos direto (o Observer), o que só compensa quando há mais de uma tela ou quando a regra muda com frequência.

---

### Referências

Mesma numeração da página do tópico.

- Reenskaug, T. *Models - Views - Controllers*. Xerox PARC, Technical Note, 1979.
- Krasner, G. E.; Pope, S. T. "A Cookbook for Using the Model-View Controller User Interface Paradigm in Smalltalk-80". *Journal of Object-Oriented Programming*, 1(3), 1988.
- Gamma, E. et al. *Design Patterns*. Addison-Wesley, 1994. Cap. 1, seção 1.2.
- Buschmann, F. et al. *Pattern-Oriented Software Architecture, Vol. 1*. Wiley, 1996. Seção 2.4.

---

Conteúdo sob CC BY-NC 4.0, uso livre com crédito. Código sob MIT.